In [30]:
import pandas as pd

# 1. Load the two datasets
df_area = pd.read_csv('final_imputed_crown_area.csv')
df_height = pd.read_csv('final_imputed_crown_height.csv')

# 2. Isolate the columns you want from the height dataset
# We keep OBJECTID so we have a "key" to match on, plus the target height column
df_height_subset = df_height[['OBJECTID', 'CROWN_HEIGHT_interpolated']]

# 3. Merge the datasets
# how='inner' tells Pandas to only keep the rows where the OBJECTID exists in BOTH datasets
# (which naturally reduces the final dataset down to your height subset size)
df_combined = pd.merge(df_area, df_height_subset, on='OBJECTID', how='inner')

# 4. Save the combined result
df_combined.to_csv('final_combined_crown_data.csv', index=False)

# Optional: Print a quick diagnostic to verify it worked
print(f"Original Area rows: {len(df_area)}")
print(f"Original Height rows: {len(df_height)}")
print(f"Combined Result rows: {len(df_combined)} (Should match the Height rows)")

#df_combined
print(df_combined.columns.tolist())


Original Area rows: 55082
Original Height rows: 1397
Combined Result rows: 1397 (Should match the Height rows)
['Unnamed: 0.1', 'Unnamed: 0', 'OBJECTID', 'SITE_NAME', 'X', 'Y', 'EXTENT_EASTING_1', 'EXTENT_NORTHING_1', 'EXTENT_EASTING_2', 'EXTENT_NORTHING_2', 'LATIN_NAME', 'COMMON_NAME', 'FULL_COMMON_NAME', 'TREE_SPECIES', 'CROWN_WIDTH', 'CROWN_HEIGHT', 'DBH', 'PLANTING_SEASON', 'CROWN_AREA_MIDPOINT', 'CROWN_AREA_COMPLETE', 'Data Source', 'CROWN_HEIGHT_interpolated']


In [31]:
columns_to_keep = [
    'OBJECTID',
    'SITE_NAME', 
    'COMMON_NAME',
    'X',
    'Y',
    'EXTENT_EASTING_1',
    'EXTENT_NORTHING_1',	
    'EXTENT_EASTING_2',
    'EXTENT_NORTHING_2',
    'LATIN_NAME',
    'COMMON_NAME',
    'FULL_COMMON_NAME', 
    'TREE_SPECIES',
    'DBH', 
    'CROWN_AREA_COMPLETE', 
    'CROWN_HEIGHT_interpolated',
]

# 3. Overwrite the dataframe with only that specific subset of columns
df_cleaned = df_combined[columns_to_keep]

df_cleaned

# Creates a temporary dataframe of just the rows missing their height
nan_rows = df_cleaned[df_cleaned['CROWN_HEIGHT_interpolated'].isna()]

print("Rows missing height data:")
print(nan_rows)

Rows missing height data:
      OBJECTID            SITE_NAME COMMON_NAME          X          Y  \
436   26634543  Newfoundland Street         NaN  359531.06  173571.96   
1047  26651091     Arnos Court Park         NaN  361205.23  171405.32   
1071  26651120     Arnos Court Park         NaN  361196.91  171424.74   
1332  26660810    Central Promenade         NaN  358591.23  172832.79   
1333  26660811    Central Promenade         NaN  358589.91  172806.52   
1334  26660812    Central Promenade         NaN  358592.46  172845.78   
1335  26660813    Central Promenade         NaN  358593.59  172859.20   
1336  26660814    Central Promenade         NaN  358590.52  172819.59   
1389  26664627    Ludlow Close Park         NaN  359670.32  174083.23   
1392  26664659    Ludlow Close Park         NaN  359657.38  174083.13   

      EXTENT_EASTING_1  EXTENT_NORTHING_1  EXTENT_EASTING_2  \
436          359531.06          173571.96         359531.06   
1047         361205.23          171405.32   

In [32]:
# --- DIAGNOSTIC: THE JUDAS TREE DOSSIER ---

# 1. Find all trees where COMMON_NAME has "Judas" in it
judas_named = df_cleaned[df_cleaned['COMMON_NAME'].str.contains('Judas', case=False, na=False)]

# 2. Find all trees where TREE_SPECIES has "Cercis siliquastrum"
cercis_species = df_cleaned[df_cleaned['TREE_SPECIES'].str.contains('Cercis siliquastrum', case=False, na=False)]

print("=== 1. TREES NAMED 'JUDAS' IN COMMON_NAME ===")
print(f"Total found: {len(judas_named)}")
if len(judas_named) > 0:
    # Print the relevant columns to see if they actually have numeric data to average!
    print(judas_named[['COMMON_NAME', 'CROWN_AREA_COMPLETE', 'CROWN_HEIGHT_interpolated']])
else:
    print("WARNING: Zero trees with 'Judas' in the COMMON_NAME column were found!")

print("\n=== 2. TREES LISTED AS 'CERCIS SILIQUASTRUM' ===")
print(f"Total found: {len(cercis_species)}")
if len(cercis_species) > 0:
    print(cercis_species[['COMMON_NAME', 'TREE_SPECIES', 'CROWN_AREA_COMPLETE', 'CROWN_HEIGHT_interpolated']])

AttributeError: 'DataFrame' object has no attribute 'str'

In [ ]:
import pandas as pd

targets = ['CROWN_AREA_COMPLETE', 'CROWN_HEIGHT_interpolated']

# --- 1. Calculate the Universal Deciduous Average ---
evergreen_set = {
    "Bay", "Box", "Cabbage Palm", "Cedar", "Chusan Palm", "Conifer", "Cypress", "Douglas Fir", "Eleagnus",
    "Eucalyptus", "Feijoa", "Fir", "Firethorn", "Hiba", "Hemlock", "Holly", "Juniper", "Laurel", "Monkey Puzzle",
    "Pine", "Pittosporum", "Redwood", "Scots Pine", "Spruce", "Strawberry Tree", "Wedding Cake Tree", "Yew"
}

# Find rows where the tree is NOT (~) in the evergreen set
deciduous_mask = ~df_cleaned['COMMON_NAME'].isin(evergreen_set)
mean_deciduous = df_cleaned.loc[deciduous_mask, targets].mean().values


# --- 2. Bulletproof the Judas Tree Average ---
# str.contains is case-insensitive (case=False) and ignores trailing spaces!
judas_mask = df_cleaned['COMMON_NAME'].str.contains('Judas', case=False, na=False)
mean_judas = df_cleaned.loc[judas_mask, targets].mean().values


# --- 3. Apply the Averages to the exact Indices ---

# Fix the Sweetgums using the Deciduous fallback (Indices 1047, 1071, 1389)
sweetgum_indices = [1047, 1071, 1389]
df_cleaned.loc[sweetgum_indices, targets] = mean_deciduous

# Fix the Cercis siliquastrum using the fuzzy Judas match (Indices 1332 to 1336)
cercis_indices = [1332, 1333, 1334, 1335, 1336]
df_cleaned.loc[cercis_indices, targets] = mean_judas


# --- 4. Verify it actually worked this time ---
print(f"Deciduous Fallback calculated as: {mean_deciduous}")
print(f"Judas Tree Average calculated as: {mean_judas}")
print("\nFinal check of the patched rows:")
print(df_cleaned.loc[sweetgum_indices + cercis_indices, targets])

Deciduous Fallback calculated as: [96.0271698   8.94898554]
Judas Tree Average calculated as: [nan nan]

Final check of the patched rows:
      CROWN_AREA_COMPLETE  CROWN_HEIGHT_interpolated
1047             96.02717                   8.948986
1071             96.02717                   8.948986
1389             96.02717                   8.948986
1332                  NaN                        NaN
1333                  NaN                        NaN
1334                  NaN                        NaN
1335                  NaN                        NaN
1336                  NaN                        NaN


In [ ]:
# Creates a temporary dataframe of just the rows missing their height
nan_rows = df_cleaned[df_cleaned['CROWN_HEIGHT_interpolated'].isna()]

print(nan_rows)

      OBJECTID          SITE_NAME COMMON_NAME          X          Y  \
1047  26651091   Arnos Court Park         NaN  361205.23  171405.32   
1071  26651120   Arnos Court Park         NaN  361196.91  171424.74   
1332  26660810  Central Promenade         NaN  358591.23  172832.79   
1333  26660811  Central Promenade         NaN  358589.91  172806.52   
1334  26660812  Central Promenade         NaN  358592.46  172845.78   
1335  26660813  Central Promenade         NaN  358593.59  172859.20   
1336  26660814  Central Promenade         NaN  358590.52  172819.59   
1389  26664627  Ludlow Close Park         NaN  359670.32  174083.23   

      EXTENT_EASTING_1  EXTENT_NORTHING_1  EXTENT_EASTING_2  \
1047         361205.23          171405.32         361205.23   
1071         361196.91          171424.74         361196.91   
1332         358591.23          172832.79         358591.23   
1333         358589.91          172806.52         358589.91   
1334         358592.46          172845.78    